In [67]:
pip install emoji

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [130]:
pip install scikit-learn

  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 8.2/8.2 MB 44.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   -- ------------------------------------- 2.6/36.6 MB 14.3 MB/s eta 0:00:03
   ---- ----------------------------------- 3.7/36.6 MB 9.2 MB/s eta 0:00:04
   ----- ---------------------------------- 5.0/36.6 MB 7.6 MB/s eta 0:00:05
   ------- -------------------------------- 6.6/36.6 MB 7.8 MB/s eta 0:00:04
   ---------- ----------------------------- 9.2/36.6 MB 8.7 MB/s eta 0:00:04
   ------------- -------------------------- 12.3/36.6 MB 9.8 MB/s eta 0:00:03
   ---------------- ----------------------- 15.2/36.6 MB 10.4 MB/s eta 0:00:03
   ------------------- -------------------- 18.1/36.6 MB 10.8 MB/s eta 0:00:02
   ---------------------- ----------------- 20.2/36.6 MB 10.7 MB/s eta 0:00:02
   ------------


[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [133]:
import numpy as np
import pandas as pd
import html
import re
import unicodedata
import emoji
from sklearn.model_selection import train_test_split
from pathlib import Path

In [69]:
df = pd.read_csv('../data/processed/tweets_clean.csv')
df.shape       

(14233, 10)

In [70]:
pd.set_option('display.max_colwidth', None)

damaged = df[df['text'].str.contains('Cancelled Flight', regex=False)]
for t in damaged['text'].sample(10, random_state=1):
    print('-', t)

- @AmericanAir flight Cancelled Flightled and was trying to reschedule but 800 number just says to call back
- @AmericanAir You have a lot on your pLate Flight dealing with Cancelled Flightlations and weather tonite. Those are more important. I know you will respond.
- @AmericanAir I understand flight Cancelled Flightation but not being able to get through to change that flight for 6 hrs is not acceptable.
- @SouthwestAir no after 75 mins of hold, I finally hung up! Ridiculous wait times. Can you assist in Cancelled Flighting reservation?
- @united you really need a better system for handling Cancelled Flightled flights http://t.co/3uBCoAsyws
- @JetBlue Cancelled Flightling flights, sending out tickets, then not telling the buyer... #jetblue how is that allowed?
- @USAirways I have been on hold with your Gold reservations line for OVER 3 HOURS now. Flight Cancelled Flightled. Trying to rebook.
- @AmericanAir has Cancelled Flighted their last 3 flight into SHV tonight &amp; all flights 

## Finding: damaged words in the tweet text


- Many tweets contain strange strings such as `Cancelled Flightled`, `Cancelled Flightlations` and `pLate Flight`. They sit inside words, so they don't read as anything a person would type.
- Examples from the data:
  - `Cancelled Flightled` = "cancelled"
  - `Cancelled Flightlations` = "cancellations"
  - `You have a lot on your pLate Flight` = "plate"
- Counts (after cleaning, 14,233 tweets): `Cancelled Flight` in 993 tweets, `Late Flight` in 446, `Flight Booking Problems` in 141. 1,516 tweets (10.7%) contain at least one.
- None of the tweets contains the plain words "cancel", "late" or "booking" in any other form. Every occurrence is damaged.

**Interpretation (an inference, not proven)**
- It looks as if the substrings "cancel", "late" and "booking" were replaced by category names, including inside other words. Who did it, and when, is unknown. The original tweets aren't available to compare.

**Why it matters**
- A model only knows the words it saw in training. This data contains no plain "cancelled", "late" or "booking", so the model would never learn them, but real users will type them.

**Decision**
- Repair the three substitutions by reversing them (`Cancelled Flight` to `cancel`, `Late Flight` to `late`, `Flight Booking Problems` to `booking`). The leftover letters rejoin the root, so `Cancelled Flightled` becomes "cancelled".
- Apply this repair to the training data only, in a separate step. It is a patch for a defect in this dataset, not a general cleaning rule, so it won't be part of the cleaning function used on live tweets.
- To check whether the repair matters, I'll train the baseline on both versions and compare them on sentences I type myself.

In [71]:
fixes={'Cancelled Flight': 'cancel',
           'Late Flight':'late',
           'Flight Booking Problems':'booking'}

def repair_dataset_artifacts(text):
    
    for bad,good in fixes.items():
        text=text.replace(bad,good)
    return text

In [72]:
repair_dataset_artifacts('@united is flight 882 Cancelled Flightled and what do I do?')

'@united is flight 882 cancelled and what do I do?'

In [73]:
df['text_step1']=df['text'].apply(repair_dataset_artifacts)

# Decoding HTML symbols

In [74]:
pd.set_option('display.max_colwidth',None)
mask=df['text_step1'].str.contains('&amp;|&lt;|&gt;')
print(mask.sum())

708


In [75]:
for t in df.loc[mask, 'text_step1'].head(4):
    print('-', t)

- @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp; they have little recourse
- @VirginAmerica I &lt;3 pretty graphics. so much better than minimal iconography. :D
- @VirginAmerica This is such a great deal! Already thinking about my 2nd trip to @Australia &amp; I haven't even gone on my 1st trip yet! ;p
- @VirginAmerica will you be making BOS&gt;LAS non stop permanently anytime soon?


In [76]:
def decode_html(text):
    decoded = html.unescape(text)
    return decoded

In [77]:
decode_html('I &lt;3 pretty graphics &amp; more')

'I <3 pretty graphics & more'

In [78]:
df['text_step2']=df['text_step1'].apply(decode_html)

In [79]:
pat = '&amp;|&lt;|&gt;'
df['text_step1'].str.contains(pat).sum()    


np.int64(708)

In [80]:

df['text_step2'].str.contains(pat).sum()    

np.int64(0)

# Removing URLs

In [81]:
pd.set_option('display.max_colwidth',None)
df.loc[df['text_step2'].str.contains('http'), 'text_step2'].head(5)

6                                     @VirginAmerica Really missed a prime opportunity for Men Without Hats parody, there. https://t.co/mWpG7grEZP
12    @VirginAmerica @virginmedia I'm flying your #fabulous #Seductive skies again! U take all the #stress away from travel http://t.co/ahlXHhKiyn
20                                                                                      @VirginAmerica I love this graphic. http://t.co/UT5GrRwAaA
33             @VirginAmerica this is great news!  America could start flights to Hawaii by end of year http://t.co/r8p2Zy3fe4 via @Pacificbiznews
34    Nice RT @VirginAmerica: Vibe with the moodlight from takeoff to touchdown. #MoodlitMonday #ScienceBehindTheExperience http://t.co/Y7O0uNxTQP
Name: text_step2, dtype: str

In [82]:
def remove_urls(text):
    return re.sub(r'http\S+', '', text)

In [83]:
remove_urls('@united thanks for the help http://t.co/abc123XYZ!')

'@united thanks for the help '

In [84]:
remove_urls('@united thanks for the help')

'@united thanks for the help'

In [85]:
df['text_step3']=df['text_step2'].apply(remove_urls)

In [86]:
df['text_step2'].str.contains('http').sum()    


np.int64(1139)

In [87]:
df['text_step3'].str.contains('http').sum()   

np.int64(0)

checking if there was only a link in tweet--> was it convereted to an empty string

In [88]:
(df['text_step3'].str.strip() == '').sum()    

np.int64(0)

In [89]:
mask = df['text_step2'].str.contains('http')
pd.DataFrame({'before': df.loc[mask, 'text_step2'],
              'after': df.loc[mask, 'text_step3']}).head(5)

,before,after
6,"@VirginAmerica Really missed a prime opportunity for Men Without Hats parody, there. https://t.co/mWpG7grEZP","@VirginAmerica Really missed a prime opportunity for Men Without Hats parody, there."
12,@VirginAmerica @virginmedia I'm flying your #fabulous #Seductive skies again! U take all the #stress away from travel http://t.co/ahlXHhKiyn,@VirginAmerica @virginmedia I'm flying your #fabulous #Seductive skies again! U take all the #stress away from travel
20,@VirginAmerica I love this graphic. http://t.co/UT5GrRwAaA,@VirginAmerica I love this graphic.
33,@VirginAmerica this is great news! America could start flights to Hawaii by end of year http://t.co/r8p2Zy3fe4 via @Pacificbiznews,@VirginAmerica this is great news! America could start flights to Hawaii by end of year via @Pacificbiznews
34,Nice RT @VirginAmerica: Vibe with the moodlight from takeoff to touchdown. #MoodlitMonday #ScienceBehindTheExperience http://t.co/Y7O0uNxTQP,Nice RT @VirginAmerica: Vibe with the moodlight from takeoff to touchdown. #MoodlitMonday #ScienceBehindTheExperience


# Remove mentions



**What the data shows**
- Every tweet contains at least one @mention (100% of 14,233), and 96.9% start with an airline handle such as `@united` or `@USAirways`.
- The handle shows who the tweet is addressed to. It says nothing about how the writer feels.

**Why the handle is a problem if kept**
- The airline is linked to the label. In my EDA, US Airways had the highest share of negative tweets (77.7%) and Virgin America the lowest. A model could learn "this airline name means negative" and use that as a shortcut instead of reading the words.
- That shortcut doesn't generalize. A tweet to an airline the model never saw would carry an unknown word, or a misleading signal.
- A word present in almost every tweet gives TF-IDF no information.
- Nothing is lost: the `airline` column still holds the airline for each tweet, so I can use it later for error analysis (for example, whether the model does worse on one airline).

**Decision:** remove all @mentions with the pattern `@\w+`, including mentions in the middle of a sentence, so the rule is the same everywhere.
- Trade-off: a mention in the middle of a sentence leaves slightly awkward wording (`thanks for` instead of `thanks @united for`). The sentiment words stay, which is what the model needs.
- This is a shared rule. The same function will clean live tweets on the website.

In [90]:
left = df['text_step3'].str.contains('@')



In [91]:
pd.set_option('display.max_colwidth',None)
df.loc[left, 'text_step3'].head(8)

0                                                                                                         @VirginAmerica What @dhepburn said.
1                                                                     @VirginAmerica I didn't today... Must mean I need to take another trip!
2                  @VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces & they have little recourse
3                                                                                     @VirginAmerica and it's a really big bad thing about it
4    @VirginAmerica seriously would pay $30 a flight for seats that didn't have this playing.\nit's really the only bad thing about flying VA
5                                                             @VirginAmerica yes, nearly every time I fly VX this “ear worm” won’t go away :)
6                                                       @VirginAmerica Really missed a prime opportunity for Men Without Hats parody, there. 
7     

In [92]:
def remove_mentions(text):
    return re.sub(r'@\w+', '', text)


In [93]:
remove_mentions('@united thanks @JetBlue for the help')

' thanks  for the help'

In [94]:
df['text_step4'] = df['text_step3'].apply(remove_mentions)

# Removing '#'

In [95]:
pd.set_option('display.max_colwidth',None)

df[df['text_step4'].str.contains("#")]['text_step4']

12                                           I'm flying your #fabulous #Seductive skies again! U take all the #stress away from travel 
15        So excited for my first cross country flight LAX to MCO I've heard nothing but great things about Virgin America. #29DaysToGo
25        What happened 2 ur vegan food options?! At least say on ur site so i know I won't be able 2 eat anything for next 6 hrs #fail
27                       amazing to me that we can't get any cold air from the vents. #VX358 #noair #worstflightever #roasted #SFOtoBOS
28                                                 LAX to EWR - Middle seat on a red eye. Such a noob maneuver. #sendambien #andchexmix
                                                                      ...                                                              
14208     I need to be at work tomorrow at 8am, therefore that doesn't help.  Direct message faster than calling 800 number? #Backwards
14215      I've been on hold for 55 mins about m

In [96]:
def clean_hashtags(text):
    return text.replace('#', '')

In [97]:
clean_hashtags('worst flight ever #disappointed #nomorevirgin')


'worst flight ever disappointed nomorevirgin'

In [98]:
clean_hashtags('call back #. Help?')


'call back . Help?'

In [99]:
df['text_step5'] = df['text_step4'].apply(clean_hashtags)

df['text_step4'].str.contains('#').sum()    # before: 2432


np.int64(2432)

In [100]:
df['text_step5'].str.contains('#').sum()    

np.int64(0)

# Normalizing the whitespaces

In [101]:
def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

In [102]:
normalize_whitespace('  thanks   for the\nhelp  ')
# 'thanks for the help'

'thanks for the help'

In [103]:
df['text_step6']=df['text_step5'].apply(normalize_whitespace)

In [104]:

df['text_step6']

0                                                                                                                           What said.
1                                                                             I didn't today... Must mean I need to take another trip!
2                          it's really aggressive to blast obnoxious "entertainment" in your guests' faces & they have little recourse
3                                                                                             and it's a really big bad thing about it
4             seriously would pay $30 a flight for seats that didn't have this playing. it's really the only bad thing about flying VA
                                                                     ...                                                              
14228                                                                                                    right on cue with the delays👌
14229      leaving over 20 minutes late. No warnings or

In [105]:
df['text_step5'].str.startswith(' ').sum()        
  

np.int64(13621)

In [106]:
df['text_step6'].str.startswith(' ').sum()       
  

np.int64(0)

In [107]:
 df['text_step6'].str.contains(r'\s{2,}').sum()    
   

np.int64(0)

In [108]:
df['text_step6'].str.contains('\n', regex=False).sum()   
   

np.int64(0)

In [109]:
(df['text_step6'] == '').sum()                    

np.int64(0)

# convert emojis and emoticons to words.

In [110]:
pd.set_option('display.max_colwidth', None)

def has_emoji(text):
    return emoji.emoji_count(text) > 0

emoji_mask = df['text'].apply(has_emoji)
print(emoji_mask.sum())          

for t in df.loc[emoji_mask, 'text'].sample(10, random_state=1):
    print('-', t)

476
- @AmericanAir now the second flight I was put on was Cancelled Flightled with no explanation!! Missed my first meeting!!#outraged😡😡
- @USAirways YOU ARE THE BEST!!! YOU ARE AMAZING!!! FOLLOW ME PLEASE;)🙏🙏🙏
- @VirginAmerica @AmericanAir can u help with flights to get a 2y old battling cancer who needs treatment in 🇺🇸 @FreyaBevan_Fund 💗 🎀 💗
- @SouthwestAir 😅 you won't let me change my reservation online so now I'm just wasting my time. http://t.co/mHA3xXaeD5
- @united I am a frequent user. Congrats u only lost my bags once in 20 years. Problem, it was today and your staff were rude!😢 still no bag!
- STAHP!! 😂😂😭😭😭😭 RT @JetBlue Our fleet's on fleek. http://t.co/BM2unRaOnI
- @united customers ✅. Customer service ❌ http://t.co/qP6Aw3nLIP
- @AmericanAir at LAX &amp; just got off the phone w/reservations. Every flight that'd get me to BOS before 11 am tmrw is apparently unavailable 😐
- @SouthwestAir props to your LAS employees working C11 gate. Because of them I am not opposed to flying t

In [111]:
emoticon_mask = df['text'].str.contains(r'(?:^|\s)[:;=]-?[()DPpOo](?:\s|$|[.!?,])', regex=True)
print(emoticon_mask.sum())
for t in df.loc[emoticon_mask, 'text'].sample(10, random_state=1):
    print('-', t)

298
- @JetBlue Yesterday on my way from EWR to FLL just after take-off. :)
#wheelsup #JetBlueSoFly http://t.co/9xkiy0Kq2j
- @USAirways The automated message isn't helpful and it's impossible to speak with a human right now. Desperately need our luggage :(
- @united I hope so too :)
- @JetBlue how do we watch the oscars though right now?! No abc on flight :( #missingtheoscars
- @USAirways Appreciate the calls when flight's delayed, but the recorded msg is tedious &amp; just goes on &amp; on. I vote for texts if practical :)
- @usairways Thx for responding :) When bags came there was mad rush towards end of jetway—why not just put thru baggage claim?
- @AmericanAir emailed AAdvantage for status match ... still waiting.  Can't wait to move from @united but help me out here :)
- @united I live abroad and am Flight Booking Problems flights for when I come visit the states :(
- @united Every United flight between Saipan and Guam is an adventure! You never know when @CapeAir's old plane will 

In [112]:
emoticons = {
    ':)': 'smile', ':-)': 'smile', ':D': 'smile', ':-D': 'smile',
    ':(': 'sad',   ':-(': 'sad',
    ';)': 'wink',  ';-)': 'wink',
    '<3': 'love',
}

def convert_emoticons(text):
    for emo, word in emoticons.items():
        pattern = r'(?<!\S)' + re.escape(emo) + r'(?=\s|$|[.!?,])'
        text = re.sub(pattern, word, text)
    return text

In [113]:
df['text_step7'] = df['text_step6'].apply(convert_emoticons)

In [114]:
def convert_emojis(text):
    text = emoji.demojize(text, delimiters=(' ', ' '), language='alias')
    text = emoji.demojize(text, delimiters=(' ', ' '))   # catches emojis with no alias
    return text.replace('_', ' ')


In [115]:
convert_emojis('so sad 😢 thanks')
# 'so sad  crying thanks'

'so sad  cry  thanks'

In [116]:
df['text_step8'] = df['text_step7'].apply(convert_emojis)
df['text_step9'] = df['text_step8'].apply(normalize_whitespace)

In [117]:
df['text_step7'].apply(has_emoji).sum()    

np.int64(473)

In [118]:
df['text_step9'].apply(has_emoji).sum()    


np.int64(0)

In [119]:
(df['text_step9'] == '').sum()            

np.int64(0)

In [120]:
df['text_step9'].str.contains(r'\s{2,}').sum()  

np.int64(0)

In [121]:
sample = df[df['text_step7'].apply(has_emoji)].sample(10, random_state=1)
for _, r in sample.iterrows():
    print(r['text_step7'], '\n->', r['text_step9'], '\n')

used to exclusively fly SW, but lately, direct flights & cheaper costs have me switched. 😔 why...so... few...direct...flights? 
-> used to exclusively fly SW, but lately, direct flights & cheaper costs have me switched. pensive why...so... few...direct...flights? 

Thanks. Which day of the week is the direct? The flights I saw went thru JFK ☀️ 
-> Thanks. Which day of the week is the direct? The flights I saw went thru JFK sunny 

can u help with flights to get a 2y old battling cancer who needs treatment in 🇺🇸 💗 🎀 💗 
-> can u help with flights to get a 2y old battling cancer who needs treatment in flag for United States heartpulse ribbon heartpulse 

. Flight 654 Delayed / Flight 409 You sold my seat / flight 0671 strand me in PHX for three hours. Thanks for the memories 😡😤😖😲😩 
-> . Flight 654 Delayed / Flight 409 You sold my seat / flight 0671 strand me in PHX for three hours. Thanks for the memories rage triumph confounded astonished weary 

STAHP!! 😂😂😭😭😭😭 RT Our fleet's on fleek. 


In [122]:
# Drop empty texts (should be 0, matching your check)
print((df['text_step9'] == '').sum())
df = df[df['text_step9'] != ''].reset_index(drop=True)

# Comparison key: lowercase, so "Thanks" and "thanks" count as the same
df['text_key'] = df['text_step9'].str.lower()

# How many rows share their text with another row?
dup_mask = df['text_key'].duplicated(keep=False)
print('rows in duplicate groups:', dup_mask.sum())
print('distinct duplicated texts:', df.loc[dup_mask, 'text_key'].nunique())

# Which duplicated texts have conflicting labels?
n_labels = df.groupby('text_key')['label'].nunique()
conflict_keys = n_labels[n_labels > 1].index
print('texts with conflicting labels:', len(conflict_keys))

0
rows in duplicate groups: 285
distinct duplicated texts: 77
texts with conflicting labels: 19


In [123]:
df.groupby('text_key')['label'].nunique().sort_values(ascending=False)

text_key
has getaway deals through may, from $59 one-way. lots of cool cities cheapflights farecompare                                     2
i have 2d and 3d embossed badges and patches superior to the ones you are currently using.                                        2
"airport snow removal method 22..." keep up the good work folks, this is where cessna's become 747's!                             2
for my grandma ella's 80th, she would love a bday greeting from your flight crew! she was a stewardess for eastern airlines.      2
's ceo battles to appease passengers and wall street -                                                                            2
                                                                                                                                 ..
“: we regret to hear this. please let us know if you need assistance. ^kn” i was finally able to secure my seat. thx              1
“: we will follow up with our maintenance team. thanks for the twee

In [124]:
df[df['text_key']=='has getaway deals through may, from $59 one-way. lots of cool cities cheapflights farecompare']

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,retweet_count,text,tweet_created,label,text_step1,text_step2,text_step3,text_step4,text_step5,text_step6,text_step7,text_step8,text_step9,text_key
108,569873669700358144,positive,0.6823,NaN,NaN,Virgin America,0,"@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/tZZJhuIbCH #CheapFlights #FareCompare",2015-02-23 06:57:25-08:00,2,"@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/tZZJhuIbCH #CheapFlights #FareCompare","@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/tZZJhuIbCH #CheapFlights #FareCompare","@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities #CheapFlights #FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities #CheapFlights #FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through may, from $59 one-way. lots of cool cities cheapflights farecompare"
109,569873668131717120,neutral,1.0000,NaN,NaN,Virgin America,0,"@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/RPdBpX3wNd #CheapFlights #FareCompare",2015-02-23 06:57:24-08:00,1,"@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/RPdBpX3wNd #CheapFlights #FareCompare","@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/RPdBpX3wNd #CheapFlights #FareCompare","@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities #CheapFlights #FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities #CheapFlights #FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through may, from $59 one-way. lots of cool cities cheapflights farecompare"
110,569873665640292353,positive,0.6806,NaN,NaN,Virgin America,0,"@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/B2Xi4YG5T8 #CheapFlights #FareCompare",2015-02-23 06:57:24-08:00,2,"@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/B2Xi4YG5T8 #CheapFlights #FareCompare","@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities http://t.co/B2Xi4YG5T8 #CheapFlights #FareCompare","@VirginAmerica has getaway deals through May, from $59 one-way. Lots of cool cities #CheapFlights #FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities #CheapFlights #FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlights FareCompare","has getaway deals through May, from $59 one-way. Lots of cool cities CheapFlig

In [125]:
conflicts = df[df['text_key'].isin(conflict_keys)]
for key, g in conflicts.groupby('text_key'):
    print(repr(key[:100]), '->', g['airline_sentiment'].value_counts().to_dict())

'"airport snow removal method 22..." keep up the good work folks, this is where cessna\'s become 747\'s' -> {'positive': 2, 'neutral': 1}
"'s ceo battles to appease passengers and wall street -" -> {'neutral': 3, 'positive': 1}
"'s ceo battles to appease passengers and wall street - waterbury republican american" -> {'neutral': 8, 'negative': 1}
'congratulations!' -> {'neutral': 1, 'positive': 1}
"for my grandma ella's 80th, she would love a bday greeting from your flight crew! she was a stewarde" -> {'positive': 1, 'neutral': 1}
'has getaway deals through may, from $59 one-way. lots of cool cities cheapflights farecompare' -> {'positive': 2, 'neutral': 2}
"i dm'd you" -> {'neutral': 1, 'negative': 1}
'i have 2d and 3d embossed badges and patches superior to the ones you are currently using.' -> {'neutral': 3, 'negative': 1}
'just sent it' -> {'negative': 1, 'neutral': 1}
'mad love nyc-' -> {'positive': 2, 'neutral': 1}
'ok thanks' -> {'positive': 1, 'neutral': 1}
"rt : our fleet's on 

##Deduplicate on cleaned text

### What we are doing
After cleaning (removing URLs, @mentions, hashtags, converting emojis), different tweets can end up with **identical text**. We find these using `text_key`, a lowercase copy of `text_step9` used only for comparison.

### Why
- **Leakage:** if one copy lands in train and an identical copy in test, the model is scored on text it has already seen, so the test score is inflated.
- **Cleaner statistics:** repeated generic phrases ("thanks", "thank you!") would otherwise be over-represented.

### What we found
- 285 rows share their text with another row, spread over 77 distinct texts (208 redundant rows).
- 19 of the 77 texts carry **conflicting labels** (mostly short generic phrases where annotators split between positive and neutral).
- The other 58 texts always have the same label (plain repeats).

### Decision rule
| Case | Action | Reason |
|---|---|---|
| Conflicting labels, clear majority (11 texts) | Relabel all copies with the majority label | Standard crowdsourcing practice (majority vote) |
| Conflicting labels, exact tie (8 texts, ~18 rows) | Drop | No basis to pick a label |
| Same label every time (58 texts) | Keep one row, drop the copies | Removes leakage |

### Why not drop every conflicting text
Texts like "thanks" (16 positive vs 1 neutral) have a clear answer. Dropping them would remove about 66 clean positive examples. For the earlier tweet_id conflicts, each pair had only 2 rows, so no majority existed and dropping was the right call.

### Trade-off
Collapsing copies means a common phrase is seen once, not many times, so the model learns less about how frequent it is. We accept this because a leak-free evaluation matters more, and the loss is small (about 208 of ~14,000 rows).

### Limitation
Near-duplicates (for example the same tweet with one extra word, or retweets with a different prefix) are not caught by exact matching.

In [126]:
tie_keys = []
majority = {}

for key in conflict_keys:
    vc = df.loc[df['text_key'] == key, 'label'].value_counts()
    if len(vc) > 1 and vc.iloc[0] == vc.iloc[1]:
        tie_keys.append(key)          
    else:
        majority[key] = vc.index[0]   

print('ties to drop:', len(tie_keys), '| majority resolved:', len(majority))


df = df[~df['text_key'].isin(tie_keys)].copy()
df['label'] = df['text_key'].map(majority).fillna(df['label']).astype(int)

print(df.shape)

ties to drop: 8 | majority resolved: 11
(14215, 20)


In [127]:
before = len(df)
df = df.drop_duplicates(subset='text_key', keep='first').reset_index(drop=True)
print(before, '->', len(df), '| removed:', before - len(df))

# check: no duplicates left
print('remaining duplicates:', df['text_key'].duplicated().sum())

# class balance after dedupe
print(df['label'].value_counts())

14215 -> 14017 | removed: 198
remaining duplicates: 0
label
0    9003
1    2863
2    2151
Name: count, dtype: int64


# Dividing into train, test and validation

In [132]:
# 70% train, 30% held out
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=42
)
# split the held-out 30% in half: 15% validation, 15% test
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42
)

print(len(train_df), len(val_df), len(test_df))
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(name, d['label'].value_counts(normalize=True).round(3).to_dict())

9811 2103 2103
train {0: 0.642, 1: 0.204, 2: 0.154}
val {0: 0.642, 1: 0.204, 2: 0.154}
test {0: 0.642, 1: 0.204, 2: 0.153}


In [134]:


out = Path('../data/processed')
out.mkdir(parents=True, exist_ok=True)

cols = ['tweet_id', 'airline', 'text', 'text_step9', 'label']
rename = {'text': 'text_raw', 'text_step9': 'text_clean'}

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d[cols].rename(columns=rename).to_csv(out / f'{name}.csv', index=False)

# check: reload and confirm shape and columns
for name in ['train', 'val', 'test']:
    check = pd.read_csv(out / f'{name}.csv')
    print(name, check.shape, list(check.columns))

train (9811, 5) ['tweet_id', 'airline', 'text_raw', 'text_clean', 'label']
val (2103, 5) ['tweet_id', 'airline', 'text_raw', 'text_clean', 'label']
test (2103, 5) ['tweet_id', 'airline', 'text_raw', 'text_clean', 'label']
